[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Your First Request


## What you will be able to do

Send a request with `requests` and read everything its response carries: the status code, the
headers, the body as bytes, text or JSON, the request that was actually sent, and any redirects on
the way. Decide when an error status should stop the program.


## The idea

### The problem

The **What an API Is** notebook sent requests with `urllib.request`, and every request took work
that was not the job: reading bytes and decoding them, parsing JSON by hand, typing a query into
the URL, and digging a `404`'s explanation out of an exception. Every program that calls an API
repeats that work, and small differences between the repetitions turn into bugs. A call that
forgets its timeout hangs, a body read twice comes back empty the second time, and a `404` crashes
a loop that should have carried on.

That work is the same for every API, so a library can do it once. For Python, the library most
code uses is `requests`.

### What requests is

> **requests** is a third-party library for HTTP. `requests.get(url)` sends a `GET` request and
> returns a **Response** object carrying everything about the response: its status code and
> headers, its body as bytes, as decoded text and as parsed JSON, the URL it came from, the
> request that was actually sent, and any redirects followed on the way.

### Why it works that way

A few decisions shape how requests behaves, and every one of them removes a kind of mistake:

- **A response is not an error.** A `404` comes back as a Response with a `status_code` of `404`,
  because to an API client "no such station" is an answer to handle. When a failure should stop
  the program, `raise_for_status()` raises.
- **The body is read once and kept.** `content`, `text` and `json()` can be used as often as you
  like. The `read()` of `urlopen` returns the body the first time and empty bytes after that.
- **The query is built for you.** `params` takes a dictionary and encodes it into the URL.
- **Header names ignore case**, as HTTP defines them, so `content-type` finds `Content-Type`.
- **There is no timeout unless you give one.** Without `timeout`, requests waits as long as the
  server takes, which can be forever, so every call in this guide passes one.
- **Redirects are followed.** A `3xx` response sends requests on to the new address, and the
  Response remembers the detour in `history`.

### Where you will meet this

requests is one of the most downloaded packages on the Python Package Index, and the Python
examples in most API documentation use it. Postman's code option offers it for any request. `httpx`,
a newer library, copies its interface closely, so what you learn here carries over. The rest of
this guide sends requests with it, and the **A Real Client** notebook wraps it in a module of your
own.

### Installing it

Colab has requests already. On your own computer, `pip install requests` adds it, inside a virtual
environment as the **Environments and pip** notebook explains.

### What this notebook covers

- The same three jobs done with `urlopen` and with requests
- The Response object: its status, its headers, and its body as bytes, text and JSON
- The request behind every response
- A query built from a dictionary with `params`, and a timeout on every call
- A redirect followed, and remembered in `history`
- `raise_for_status`, for when an error status should stop the program
- Four errors, from a name `urlopen` taught to a `404` read as a station

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests

response = requests.get("http://127.0.0.1:8765/stations/tromso", timeout=10)

print(response.status_code, response.reason)
print(response.headers["Content-Type"])
print(response.json())
```

```
200 OK
application/json
{'id': 'tromso', 'name': 'Tromso', 'latitude': 69.65, 'longitude': 18.96}
```

That is the `GET` request the **What an API Is** notebook sent, with the status code, a header and
the parsed body each a single attribute or method call away.


## Setup

Six imports, the last of them the practice API.

- `json` and `urllib.request` repeat the **What an API Is** notebook's approach in the first
  example, for comparison. `urllib.request` also fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `HTTPError` is how `urlopen` reports a `404`, needed for the same comparison
- `requests` sends every other request in this notebook
- `practice_api` is the server this guide talks to, started in the background by `start()`


In [1]:
import json
import urllib.request
from pathlib import Path
from urllib.error import HTTPError

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if not Path("practice_api.py").exists():      # true in Colab, which starts with only the notebook
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")

import practice_api

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### Before and after: three jobs, with urlopen and with requests

Three jobs the **What an API Is** notebook did with `urlopen`: read a station's name, read the
reason a `404` gives, and fetch three days of Tromso's temperatures with a query. First, the way
that notebook did them:


In [2]:
# With urlopen: three jobs, as the What an API Is notebook did them.

# 1. Read a station's name. Correct result: Tromso.
with urllib.request.urlopen(f"{BASE}/stations/tromso") as response:
    station = json.loads(response.read())
print("1.", station["name"])

# 2. Read the reason a 404 gives. Correct result: 404, and the error message, which urlopen
#    only hands over inside an exception.
try:
    urllib.request.urlopen(f"{BASE}/stations/atlantis")
except HTTPError as error:
    print("2.", error.code, json.loads(error.read())["error"])

# 3. Fetch three days of temperatures, with the query typed into the URL.
#    Correct result: [4.6, 6.4, 7.1].
url = ("https://archive-api.open-meteo.com/v1/archive?latitude=69.65&longitude=18.96"
       "&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_mean&models=era5")
with urllib.request.urlopen(url, timeout=30) as response:
    print("3.", json.loads(response.read())["daily"]["temperature_2m_mean"])


1. Tromso
2. 404 no station with id 'atlantis'


3. [4.6, 6.4, 7.1]


Now the same three jobs with requests:


In [3]:
# With requests: the same three jobs.

# 1. Read a station's name. Correct result: Tromso.
response = requests.get(f"{BASE}/stations/tromso", timeout=10)
print("1.", response.json()["name"])

# 2. Read the reason a 404 gives. Correct result: 404, and the error message, from an
#    ordinary response.
response = requests.get(f"{BASE}/stations/atlantis", timeout=10)
print("2.", response.status_code, response.json()["error"])

# 3. Fetch three days of temperatures, with requests building the query from a dictionary.
#    Correct result: [4.6, 6.4, 7.1].
response = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=30, params={
    "latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15", "end_date": "2025-01-17",
    "daily": "temperature_2m_mean", "models": "era5"})
print("3.", response.json()["daily"]["temperature_2m_mean"])


1. Tromso
2. 404 no station with id 'atlantis'


3. [4.6, 6.4, 7.1]


The same three lines of output. The difference is in what was no longer needed: no `with` block, no
bytes to decode, no `try` around a `404`, and no query typed by hand. Every call still passes
`timeout`, for a reason a later section explains.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.

### The response object

`requests.get` returns a `Response`. Its first attributes say how the request went:


In [4]:
response = requests.get(f"{BASE}/stations/tromso", timeout=10)

print(type(response))
print("status_code:", response.status_code)
print("reason:     ", response.reason)
print("ok:         ", response.ok)
print("url:        ", response.url)


<class 'requests.models.Response'>
status_code: 200
reason:      OK
ok:          True
url:         http://127.0.0.1:8765/stations/tromso


`status_code` is the number from the status line, and `reason` is its phrase. `ok` is `True` for
any status code below 400, a quick test that the request worked. `url` is where the response came
from, which is not always the URL you asked for, as the section on redirects shows.

### Headers, in any capitalization


In [5]:
print(response.headers)
print(response.headers["Content-Type"])
print(response.headers["content-type"])
print(response.headers.get("Retry-After"))


{'Server': 'PracticeAPI/1.0', 'Date': 'Sun, 01 Mar 2026 09:00:00 GMT', 'Content-Type': 'application/json', 'Content-Length': '73'}
application/json
application/json
None


`headers` prints like a dictionary and behaves like one, with a single difference that HTTP
requires: header names ignore case, so `content-type` finds `Content-Type`. `get` returns `None` for
a header the response does not carry, here `Retry-After`, which the **Rate Limits** notebook
watches for. The **Headers and Content Types** notebook reads many more.

### The body, three ways


In [6]:
print("content: ", response.content)
print("text:    ", response.text)
print("json():  ", response.json())
print("encoding:", response.encoding)


content:  b'{"id": "tromso", "name": "Tromso", "latitude": 69.65, "longitude": 18.96}'
text:     {"id": "tromso", "name": "Tromso", "latitude": 69.65, "longitude": 18.96}
json():   {'id': 'tromso', 'name': 'Tromso', 'latitude': 69.65, 'longitude': 18.96}
encoding: utf-8


- `content` is the body exactly as it arrived, as bytes.
- `text` is `content` decoded into a string with `encoding`, which requests takes from the
  `Content-Type` header. For JSON sent without a character set, as here, it uses UTF-8.
- `json()` parses `text` into Python values. It is a method, so it needs its parentheses.

requests keeps the body, so all three can be read as often as you like. `urlopen` does not:


In [7]:
with urllib.request.urlopen(f"{BASE}/stations/tromso") as raw:
    first, second = raw.read(), raw.read()

print("first read:  ", len(first), "bytes")
print("second read: ", second)
print("json() twice:", response.json() == response.json())


first read:   73 bytes
second read:  b''
json() twice: True


A second `read()` finds nothing left, which is a quiet way to lose a body in code that reads it in
two places.

### The request behind the response


In [8]:
sent = response.request

print("method: ", sent.method)
print("url:    ", sent.url)
print("headers:", {name: sent.headers[name] for name in ("Accept", "Connection")})
print("client: ", sent.headers["User-Agent"].split("/")[0])


method:  GET
url:     http://127.0.0.1:8765/stations/tromso
headers: {'Accept': '*/*', 'Connection': 'keep-alive'}
client:  python-requests


`response.request` is the request exactly as requests sent it, headers included, which makes it the
first place to look when an API responds in a way you did not expect. requests adds headers of its
own. `Accept: */*` says any content type will do, `Connection: keep-alive` asks the server to keep
the connection open for the next request, and `User-Agent` names the client as `python-requests`
followed by its version.

The version depends on your installation, which is why only the name is printed here. For the same
reason this cell leaves out `Accept-Encoding`, which changes with the compression libraries
installed. Some APIs insist on a `User-Agent`: GitHub's refuses a request that has none.

### A query built from a dictionary, and a timeout on every call


In [9]:
weather = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={"latitude": 69.65, "longitude": 18.96, "start_date": "2025-01-15",
            "end_date": "2025-01-17", "daily": "temperature_2m_mean", "models": "era5"},
    timeout=30,
)

print(weather.url)
print(weather.status_code, weather.headers["Content-Type"])
print(weather.json()["daily"])


https://archive-api.open-meteo.com/v1/archive?latitude=69.65&longitude=18.96&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_mean&models=era5
200 application/json; charset=utf-8
{'time': ['2025-01-15', '2025-01-16', '2025-01-17'], 'temperature_2m_mean': [4.6, 6.4, 7.1]}


`params` turned the dictionary into the query, and `weather.url` shows the URL requests actually
sent. The **Query Parameters** notebook covers `params` in full, including the values that need
encoding.

`timeout=30` sets how long requests waits: up to 30 seconds to connect, and then up to 30 seconds
between one piece of the response and the next. It is not a limit on the whole request. Without it,
a server that accepts the connection and never responds holds the program forever. The **Errors
and Retries** notebook shows what to do when the time runs out.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.

### A redirect, followed and remembered

The practice API's stations once lived under `/v0/`, and the old addresses still work: they respond
`301 Moved Permanently`, and name the new address in a `Location` header. Retired addresses like
these are rarely documented, and the practice API's OpenAPI document leaves them out too.


In [10]:
moved = requests.get(f"{BASE}/v0/stations/tromso", timeout=10)

print(moved.status_code, moved.url)
print("history:", moved.history)
print("the first response:", moved.history[0].status_code, moved.history[0].headers["Location"])


200 http://127.0.0.1:8765/stations/tromso
history: [<Response [301]>]
the first response: 301 /stations/tromso


requests followed the redirect on its own. The response you get is the `200` from the new address,
`url` shows where it ended up, and `history` keeps every response on the way, here the `301`. To
see the redirect itself and stop there, pass `allow_redirects=False`:


In [11]:
first = requests.get(f"{BASE}/v0/stations/tromso", allow_redirects=False, timeout=10)

print(first.status_code, first.reason, first.headers["Location"], first.is_redirect)


301 Moved Permanently /stations/tromso True


Following redirects without saying so is convenient, and occasionally surprising: a program can read
a different address from the one in its source for years without anyone noticing. Check `history`
when a response is not what its URL suggests.

### raise_for_status, for when an error should stop the program


In [12]:
missing = requests.get(f"{BASE}/stations/atlantis", timeout=10)

print(missing.status_code, missing.reason, missing.ok)
print(missing.json())


404 Not Found False
{'error': "no station with id 'atlantis'"}


No exception: a `404` is an ordinary response, with a body you can read like any other. Often that
is right, as when a missing station should simply be skipped. When an error status means the
program cannot carry on, `raise_for_status()` raises `requests.HTTPError` for any status of 400 or
above, and does nothing otherwise:


In [13]:
for station_id in ["tromso", "atlantis"]:
    response = requests.get(f"{BASE}/stations/{station_id}", timeout=10)
    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        print(f"{station_id}: failed, {error}")
    else:
        print(f"{station_id}: {response.json()['name']}")


tromso: Tromso
atlantis: failed, 404 Client Error: Not Found for url: http://127.0.0.1:8765/stations/atlantis


The message names the status code, the reason and the URL, which is most of what a failure report
needs. The **Status Codes** notebook decides, code by code, which failures should stop a program.

### A report from two APIs, with requests

The **What an API Is** notebook ended by asking both APIs a question: where every station is, and
what the weather was there. Here is that program with requests, plus a check the first version
could not make easily: every response must succeed and must be JSON, or the program stops with a
message saying which.


In [14]:
def get_json(url, **params):
    """GET a URL and return its JSON body, stopping on an error status or a body that is not JSON."""
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    if not response.headers["Content-Type"].startswith("application/json"):
        raise ValueError(f"{response.url} sent {response.headers['Content-Type']}, not JSON")
    return response.json()


for summary in get_json(f"{BASE}/stations"):
    station = get_json(f"{BASE}/stations/{summary['id']}")
    daily = get_json("https://archive-api.open-meteo.com/v1/archive",
                     latitude=station["latitude"], longitude=station["longitude"],
                     start_date="2025-01-15", end_date="2025-01-17",
                     daily="temperature_2m_mean", models="era5")["daily"]
    print(f"{station['name']:<9} {daily['temperature_2m_mean']}")


Bergen    [7.8, 8.0, 8.0]


Oslo      [0.0, 2.3, 3.0]


Svalbard  [-12.9, -12.8, -13.8]


Tromso    [4.6, 6.4, 7.1]


### Where each part came from

| In the program | What it relies on | The section that showed it |
|---|---|---|
| `requests.get(url, params=params, timeout=30)` | a query built from a dictionary, and a timeout on every call | A query built from a dictionary, and a timeout on every call |
| `response.raise_for_status()` | an error status stops the program with a clear message | raise_for_status, for when an error should stop the program |
| `response.headers["Content-Type"]` | a header found in any capitalization | Headers, in any capitalization |
| `response.json()` | the body, parsed | The body, three ways |
| `f"{BASE}/stations/{summary['id']}"` | a path parameter for every station | the **What an API Is** notebook |

The same four lines as the **What an API Is** notebook's version, from a program that checks what it
receives. Pointed at the practice API's home page, which is HTML, `get_json` stops and says why:


In [15]:
try:
    get_json(BASE)
except ValueError as error:
    print(error)


http://127.0.0.1:8765/ sent text/html; charset=utf-8, not JSON


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/03-your-first-request-solutions.ipynb).

**1.** Use requests to fetch `/stations/svalbard`, and print its status code, its `Content-Length`
header and the station's name.


In [16]:
# your code here


**2.** Fetch `/stations/oslo`, and print the body three ways, `content`, `text` and `json()`, with
the type of each.


In [17]:
# your code here


**3.** Fetch `/stations`, and print the method, the URL and the `Accept` header of the request
behind the response.


In [18]:
# your code here


**4.** Use `params` to ask Open-Meteo for Bergen's three daily mean temperatures in Fahrenheit, for
the same dates, and print the URL requests sent and the values. Bergen is at latitude `60.39` and
longitude `5.32`, and the unit parameter is `temperature_unit`.


In [19]:
# your code here


**5.** Fetch the old address `/v0/stations/bergen`, and print the final URL, the status code of
every response in `history`, and the final status code.


In [20]:
# your code here


**6.** Write `station_name(station_id)`, which returns a station's name, returns `None` when the
practice API responds `404`, and raises for any other error status. Try it on `bergen` and
`narvik`.


In [21]:
# your code here


## Common errors

### AttributeError: 'Response' object has no attribute 'status'

The responses from `urlopen` call it `status`, so the habit carries over:


In [22]:
response = requests.get(f"{BASE}/stations/tromso", timeout=10)
response.status


AttributeError: 'Response' object has no attribute 'status'

requests names it `status_code`. The two libraries were written separately and chose different
names for the same number, which is worth remembering when code moves from one to the other.


In [23]:
print(response.status_code)


200


### TypeError: 'method' object is not subscriptable


In [24]:
response.json["name"]


TypeError: 'method' object is not subscriptable

`json` is a method. Without parentheses, `response.json` is the method itself rather than the parsed
body, so there is nothing to look a name up in. Call it:


In [25]:
print(response.json()["name"])


Tromso


### TypeError: string indices must be integers, not 'str'


In [26]:
response.text["name"]


TypeError: string indices must be integers, not 'str'

`text` is the body as a string, and a string is indexed by position, so `"name"` means nothing to
it. The JSON has not been parsed yet, and `json()` parses it:


In [27]:
print(type(response.text).__name__, "->", type(response.json()).__name__, response.json()["name"])


str -> dict Tromso


### KeyError: 'name', from a 404 read as a station


In [28]:
narvik = requests.get(f"{BASE}/stations/narvik", timeout=10)
narvik.json()["name"]


KeyError: 'name'

The request raised nothing, and the body parsed without complaint, because it was the practice
API's `404`, whose JSON holds `error` rather than `name`. So the `KeyError` arrives a step after the
real problem, and points at the line that read the body rather than at the request that failed.
Check the status before reading the body:


In [29]:
if narvik.ok:
    print(narvik.json()["name"])
else:
    print("no such station:", narvik.status_code, narvik.json()["error"])


no such station: 404 no station with id 'narvik'


Or call `narvik.raise_for_status()` first, when a missing station should stop the program instead.


## Recap

- `requests.get(url, params=..., timeout=...)` sends a `GET` request and returns a `Response`.
- `status_code`, `reason` and `ok` say how the request went. A `404` is an ordinary response, not an
  exception.
- `headers` finds a header in any capitalization, and `get` returns `None` for a header that is not
  there.
- The body comes three ways: `content` as bytes, `text` decoded with `encoding`, and `json()` parsed.
  requests keeps it, so it can be read more than once.
- `response.request` is the request as sent, `url` is where the response came from, and `history`
  lists any redirects followed on the way.
- `raise_for_status()` raises `requests.HTTPError` for a status of 400 or above, for when an error
  should stop the program.
- Pass `timeout` on every call. Without it, requests waits as long as the server takes.
- It is `status_code`, not `status`, and `json()` needs its parentheses.


## What is next

The **Status Codes** notebook. Every response here was a `200`, a `301` or a `404`. That notebook
meets the codes a server sends when a request is wrong in other ways, when the server itself fails,
and when a client sends too many requests, and decides what code should do about each.


---

&#8592; **Previous:** [Exploring an API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/02-exploring-an-api.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
